# Full connectivity DISU with `vorflow.disu`

This notebook builds fully unstructured MODFLOW 6 grids (DISU) from vorflow meshes and FloPy-style layer arrays.

1. **A dipping aquifer.** This is the test of Provost et al. (2025). With DISV-style ("layered") connectivity, flow cannot follow a dipping layer. Connecting every pair of cells whose faces overlap vertically ("full" connectivity) fixes it, with XT3D.
2. **A Voronoi model refined at depth.** A pit area gets smaller cells only in the lower layers (`disu.splice`). The layers dip, and a well sits in the pit.

The full-connectivity algorithm follows C. D. Langevin's USGS `disv2disu.py`, published with:

- Provost, A.M., Bardot, K., Langevin, C.D. and McCallum, J.L. (2025), *Groundwater* 63(3), 399-408, <https://doi.org/10.1111/gwat.13459>.

See also Bardot et al. (2023), <https://doi.org/10.1111/gwat.13273>, and [LoopFlopy](https://github.com/kerrybardot/loopflopy).

**Requirements:**

- `python -m pip install -e ".[examples]"`;
- the MODFLOW 6 executable on PATH, or an `MF6_EXE` path (`get-modflow /path/to/bin --subset mf6`).

In [ ]:
import os
import shutil
import tempfile
from pathlib import Path

import geopandas as gpd
import numpy as np
from shapely.geometry import LineString, box

from vorflow import ConceptualMesh, MeshGenerator, VoronoiTessellator, disu

# Import the meshing package before FloPy and its plotting backend.
import flopy
import matplotlib.pyplot as plt

mf6_exe = shutil.which(os.environ.get("MF6_EXE", "mf6"))
if mf6_exe is None:
    raise FileNotFoundError("MODFLOW 6 executable not found; set MF6_EXE or add mf6 to PATH.")
workspace = Path(tempfile.mkdtemp(prefix="vorflow_disu_"))

## 1. A dipping aquifer

The setup is a cross-section of 11 one-metre columns. A 3-layer aquifer (K = 1) dips at 30 degrees between 3 + 3 layers of low K (1e-4). The analytic head `h = -x cos(30°) - z sin(30°)` gives a flux of 1 m/d along the aquifer, and it is fixed on the perimeter. The plan grid here is a row of squares, but any vorflow grid works the same way.

In [ ]:
def strip(n):
    polygons = [box(i, 0, i + 1, 1) for i in range(n)]
    return gpd.GeoDataFrame({"x": [i + 0.5 for i in range(n)], "y": 0.5}, geometry=polygons)


def run(name, d, k, chd, xt3d):
    sim = flopy.mf6.MFSimulation(sim_ws=str(workspace / name), exe_name=mf6_exe)
    flopy.mf6.ModflowTdis(sim)
    flopy.mf6.ModflowIms(sim, outer_dvclose=1e-9, inner_dvclose=1e-10, rcloserecord=1e-9,
                         linear_acceleration="bicgstab", inner_maximum=500)
    gwf = flopy.mf6.ModflowGwf(sim, modelname="m", save_flows=True)
    flopy.mf6.ModflowGwfdisu(gwf, **d.get_gridprops_disu6())
    # With offset layers, XT3D needs ANGLE2/ANGLE3 (zeros are fine) to use the vertical offsets.
    angles = dict(angle2=0.0, angle3=0.0) if xt3d else {}
    flopy.mf6.ModflowGwfnpf(gwf, k=k, k33=k, xt3doptions=xt3d, save_specific_discharge=True, **angles)
    flopy.mf6.ModflowGwfic(gwf, strt=float(np.mean([h for _, h in chd])))
    flopy.mf6.ModflowGwfchd(gwf, stress_period_data=chd)
    flopy.mf6.ModflowGwfoc(gwf, head_filerecord="m.hds", budget_filerecord="m.cbc",
                           saverecord=[("HEAD", "ALL"), ("BUDGET", "ALL")])
    sim.write_simulation(silent=True)
    success, output = sim.run_simulation(silent=True, report=True)
    assert success, "\n".join(output[-20:])
    return gwf


dip = np.radians(30)
grid = strip(11)
rise = np.tan(dip) * grid.x.to_numpy()
surfaces = rise[None, :] + np.arange(6, -4, -1)[:, None]      # 10 surfaces -> 9 layers of 1 m
results = {}
for connectivity, xt3d in [("layered", True), ("full", False), ("full", True)]:
    d = disu.DisuGrid(grid, surfaces[0], surfaces[1:], connectivity=connectivity).build()
    cm = d.cellmap[d.cellmap.node >= 0]
    x, layer = cm.x.to_numpy(), cm.layer.to_numpy()
    z = ((cm.top + cm.bot) / 2).to_numpy()
    exact = -x * np.cos(dip) - z * np.sin(dip)
    edge = (x < 1) | (x > 10) | (layer == 0) | (layer == 8)
    aquifer = (layer >= 3) & (layer <= 5)
    k = np.where(aquifer, 1.0, 1e-4)
    chd = [[(int(n),), float(exact[n])] for n in np.flatnonzero(edge)]
    gwf = run(f"dip_{connectivity}_{xt3d}", d, k, chd, xt3d)
    spdis = gwf.output.budget().get_data(text="DATA-SPDIS")[0]
    centre = np.flatnonzero(aquifer & (np.abs(x - 5.5) < 0.1) & (layer == 4))[0]
    q = np.hypot(spdis["qx"][centre], spdis["qz"][centre])
    angle = np.degrees(np.arctan2(spdis["qz"][centre], spdis["qx"][centre]))
    results[connectivity, xt3d] = d
    print(f"{connectivity:8s} XT3D={xt3d!s:5s}  |q| = {q:.3f} m/d (1.000)  direction = {angle:6.2f} deg (30.00)")

Layered connectivity forces the flow to be horizontal, whatever the formulation. Full connectivity with XT3D gives the analytic flux. The report shows what the grid looks like: the offset ratio (offset / cell thickness) is the quantity Provost et al. use to explain when layers lose contact.

In [ ]:
print(results["full", True].report())

## 2. A Voronoi model refined at depth

The domain is 1000 m × 600 m with 60 m cells. A 200 m pit gets 12 m cells only in the three lower layers. The pit outline is a **barrier line in both meshes**, so it is made of cell faces in both, and `disu.splice` can keep the coarse cells outside and the fine cells inside. The nodes along the outline do not need to match: DISU connects cells through the shared face segments.

In [ ]:
domain = box(0, 0, 1000, 600)
pit = box(400, 200, 600, 400)


def mesh(pit_resolution=None):
    cm = ConceptualMesh()
    cm.add_polygon(domain, zone_id=1, resolution=60)
    cm.add_line(LineString(pit.exterior.coords), line_id="pit", resolution=30, is_barrier=True)
    if pit_resolution:
        cm.add_polygon(pit, zone_id=2, resolution=pit_resolution, embed=False)
    mg = MeshGenerator(background_lc=60, conceptual_mesh=cm, verbosity=0)
    mg.generate()
    return VoronoiTessellator(mg, cm).generate()


base = mesh()
pit_grid = disu.splice(base, mesh(pit_resolution=12), pit)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, g, title in zip(axes, (base, pit_grid), ("layers 1-2: base grid", "layers 3-5: refined in the pit")):
    g.boundary.plot(ax=ax, linewidth=0.4, color="k")
    ax.set_title(f"{title} ({len(g)} cells)")
    ax.set_aspect("equal")

The layers dip 3 % towards the east. Each layer's `botm` is given on that layer's grid. For a cell that exists only in the pit grid, the top is taken from the bottom of the coarse cell above it.

In [ ]:
grids = [base, base, pit_grid, pit_grid, pit_grid]
depths = (20.0, 40.0, 50.0, 60.0, 80.0)
top = 100.0 - 0.03 * base.x.to_numpy()
botm = [100.0 - depth - 0.03 * g.x.to_numpy() for g, depth in zip(grids, depths)]
d = disu.DisuGrid(grids, top, botm).build()
print(d.report())

Fixed heads go on the west (60 m) and east (50 m) edges. A well pumps 500 m³/d from the deepest layer in the middle of the pit. `locate` finds its node from coordinates, and `to_layers` maps the heads back to each layer's grid for plotting.

In [ ]:
cm = d.cellmap[d.cellmap.node >= 0]
west = cm.node[cm.x < 30].to_numpy()
east = cm.node[cm.x > 970].to_numpy()
chd = [[(int(n),), 60.0] for n in west] + [[(int(n),), 50.0] for n in east]
well = d.locate(500.0, 300.0, 100.0 - 75.0 - 0.03 * 500.0)
print("well node", well, cm.set_index("node").loc[well, ["layer", "x", "y", "top", "bot"]].to_dict())

sim = flopy.mf6.MFSimulation(sim_ws=str(workspace / "pit"), exe_name=mf6_exe)
flopy.mf6.ModflowTdis(sim)
flopy.mf6.ModflowIms(sim, outer_dvclose=1e-8, inner_dvclose=1e-9, linear_acceleration="bicgstab",
                     inner_maximum=500)
gwf = flopy.mf6.ModflowGwf(sim, modelname="pit")
flopy.mf6.ModflowGwfdisu(gwf, **d.get_gridprops_disu6())
flopy.mf6.ModflowGwfnpf(gwf, k=5.0, k33=0.5, xt3doptions=True, angle2=0.0, angle3=0.0)
flopy.mf6.ModflowGwfic(gwf, strt=55.0)
flopy.mf6.ModflowGwfchd(gwf, stress_period_data=chd)
flopy.mf6.ModflowGwfwel(gwf, stress_period_data=[[(int(well),), -500.0]])
flopy.mf6.ModflowGwfoc(gwf, head_filerecord="pit.hds", saverecord=[("HEAD", "ALL")])
sim.write_simulation(silent=True)
success, output = sim.run_simulation(silent=True, report=True)
assert success, "\n".join(output[-20:])
heads = d.to_layers(gwf.output.head().get_data().ravel())   # one array per layer

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, layer in zip(axes, (1, 4)):
    g = d.grids_per_layer[layer].assign(head=heads[layer])
    g.plot(column="head", ax=ax, legend=True, cmap="viridis")
    ax.set_title(f"layer {layer + 1} heads")
    ax.set_aspect("equal")

**Notes:**

- **`cellmap`** lists every cell with its node (-1 when removed), layer, grid, `node_id`, status and elevations. `save_cellmap()` writes it to CSV for post-processing without vorflow. MODFLOW 6 DISU accepts IDOMAIN 0/1 only, so it cannot carry such tags.
- **Pinch-outs:** idomain -1 or `min_thickness` cells are removed, and their interval goes to the cells above and below (`collapse="split"`).
- **MODFLOW-USG:** `get_gridprops_disu5()` and `write_gsf()` give the inputs (no XT3D there).
- **Existing models:** `disu.DisuGrid.from_disv(vertices, cell2d, top, botm, idomain)` reads any DISV model, for example one made with LoopFlopy.